# FACTS2 BAM + fair-temperature, ssp585: results

What drives what: **fair-temperature** simulates 500 warming trajectories, **bamber19-icesheets** turns each into
ice-sheet sea-level contributions (East/West Antarctica, Greenland), and **facts-total** sums them.
All values are projections for 2020-2150 relative to a 2005 baseline, in **mm** (500 samples).

Produced by `notebooks/run_bam.py`; this notebook only reads the netCDF outputs, so it runs in seconds and needs
`xarray`, `netcdf4`, `matplotlib`, `pandas` (no containers).

**Reading the charts.** Lines are the median; the dark band is the 17-83% range ("likely"), the light band 5-95%.
Colour follows the entity everywhere: blue = East Antarctica, orange = West Antarctica, aqua = Greenland.

In [ ]:
import json, os
from pathlib import Path
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

WS  = Path(os.environ.get("FACTS2_WORKSPACE", "../facts2")).resolve()
OUT = WS / "apptainer_experiments/bam_ssp585/output-apptainer"
FIG = os.environ.get("FIG_DIR")          # optional: also save PNGs here
if FIG: Path(FIG).mkdir(parents=True, exist_ok=True)

# Palette: reference categorical slots 1-3 (fixed order, never cycled) + neutral inks. Light surface.
C = {"EAIS": "#2a78d6", "WAIS": "#eb6834", "GIS": "#1baf7a"}
INK, INK2, MUTED, GRID, TOTAL = "#0b0b0b", "#52514e", "#8a8985", "#e4e3df", "#0b0b0b"
NAMES = {"EAIS": "East Antarctica", "WAIS": "West Antarctica", "GIS": "Greenland"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160, "font.size": 10.5, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.titleweight": "semibold",
    "axes.titlesize": 12, "axes.titlelocation": "left", "legend.frameon": False,
})
PCT = [5, 17, 50, 83, 95]

def slr(mod, name, var="sea_level_change"):
    ds = xr.open_dataset(OUT / mod / name)
    return ds["years"].values, ds[var].values.squeeze()          # (years,), (samples, years)

def fan(ax, x, a, color, label=None, lw=2.0, alpha=(0.16, 0.30)):
    p5, p17, p50, p83, p95 = np.percentile(a, PCT, axis=0)
    ax.fill_between(x, p5, p95, color=color, alpha=alpha[0], lw=0)
    ax.fill_between(x, p17, p83, color=color, alpha=alpha[1], lw=0)
    ax.plot(x, p50, color=color, lw=lw, label=label)
    return p50

def save(fig, name):
    if FIG: fig.savefig(Path(FIG) / f"{name}.png", bbox_inches="tight", facecolor="white")

yrs, _ = slr("bamber19-icesheets", "GIS-gslr.nc")
comp_g = {k: slr("bamber19-icesheets", f"{k}-gslr.nc")[1] for k in C}
comp_l = {k: slr("bamber19-icesheets", f"{k}-lslr.nc")[1] for k in C}
ais_g, gis_g = comp_g["EAIS"] + comp_g["WAIS"], comp_g["GIS"]
print(f"{comp_g['GIS'].shape[0]} samples, {yrs[0]}-{yrs[-1]}, {len(yrs)} time steps; outputs: {OUT}")

## 1. Warming scenario
The climate input: global-mean surface-air-temperature change under ssp585 from the FaIR ensemble. Everything below
inherits this spread.

In [ ]:
g = xr.open_dataset(OUT / "fair-temperature/gsat.nc")
gy = g["years"].values
sel = (gy >= 1990) & (gy <= 2150)
gs = g["surface_temperature"].values.squeeze()[:, sel]
unit = g["surface_temperature"].attrs.get("units", "deg C")

fig, ax = plt.subplots(figsize=(8, 3.8))
med = fan(ax, gy[sel], gs, INK2, lw=2.2)
ax.axvspan(2020, 2150, color=GRID, alpha=0.35, lw=0, zorder=0)
ax.text(2021, ax.get_ylim()[1]*0.97, "BAM projection window", color=MUTED, fontsize=9, va="top")
ax.annotate(f"{med[-1]:.1f} {unit} (median, 2150)", (gy[sel][-1], med[-1]), xytext=(-6, 10),
            textcoords="offset points", ha="right", color=INK, fontsize=10)
ax.set(title="Global surface temperature, ssp585", ylabel=f"GSAT change ({unit})")
ax.set_xlim(1990, 2150)
fig.tight_layout(); save(fig, "1_gsat"); plt.show()

## 2. Ice-sheet contributions through time
Three panels share one y-axis so sizes are directly comparable. Greenland is the steadier contributor; the Antarctic
sheets carry the long upper tail.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True)
for ax, k in zip(axes, C):
    p50 = fan(ax, yrs, comp_g[k], C[k])
    ax.set_title(NAMES[k]); ax.set_xlim(yrs[0], yrs[-1]); ax.set_xticks([2020, 2060, 2100, 2150])
    ax.annotate(f"{p50[-1]:.0f} mm", (yrs[-1], p50[-1]), xytext=(-4, 8), textcoords="offset points",
                ha="right", color=INK, fontsize=10)
axes[0].set_ylabel("Global sea-level contribution (mm)")
fig.tight_layout(); save(fig, "2_components"); plt.show()

## 3. Where the uncertainty is - 2150 distributions
The same data at the final year as full distributions. Bars mark 5th / 50th / 95th percentiles. Antarctic outcomes
are strongly skewed: a small share of samples drives the extreme end.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
data = [comp_g[k][:, -1] for k in C]
parts = ax.violinplot(data, showextrema=False, widths=0.8)
labels = []
for body, k in zip(parts["bodies"], C):
    body.set_facecolor(C[k]); body.set_alpha(0.35); body.set_edgecolor(C[k])
for i, (k, d) in enumerate(zip(C, data), start=1):
    q5, q50, q95 = np.percentile(d, [5, 50, 95])
    ax.vlines(i, q5, q95, color=C[k], lw=2.5)
    ax.plot(i, q50, "o", color=C[k], mec="white", mew=2, ms=9, zorder=3)
    labels.append(f"{NAMES[k]}\nmedian {q50:.0f} mm\n5-95%: {q5:.0f} to {q95:.0f}")
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_xticks([1, 2, 3], labels, fontsize=9.5); ax.grid(axis="x", visible=False)
ax.set(title=f"Sea-level contribution in {yrs[-1]}", ylabel="Global sea-level contribution (mm)")
fig.tight_layout(); save(fig, "3_distributions"); plt.show()

## 4. What the New York site adds: a fixed factor per ice sheet
`location.lst` holds one coastal site (40.7 N, -74 W). Ice loss redistributes gravity and shifts Earth's shape, so local
sea-level change differs from the global mean. In this module the local result is the global result times a constant
**fingerprint factor** per ice sheet (identical for all 500 samples), so the factor is the whole story at this site:
West Antarctic melt raises New York about 22% *more* than the global mean, East Antarctic about 4% more, and nearby
Greenland melt only **44%** as much.

In [ ]:
fac = {k: float(np.median(comp_l[k][np.abs(comp_g[k]) > 50] / comp_g[k][np.abs(comp_g[k]) > 50])) for k in C}
spread = max(float(np.std(comp_l[k][np.abs(comp_g[k]) > 50] / comp_g[k][np.abs(comp_g[k]) > 50])) for k in C)
assert spread < 1e-3, "local is no longer a fixed multiple of global; this chart would be wrong"

fig, ax = plt.subplots(figsize=(8, 3.2))
ks = list(C)[::-1]
ax.barh(range(len(ks)), [fac[k] for k in ks], color=[C[k] for k in ks], height=0.55)
ax.axvline(1, color=INK2, lw=1.2, ls=(0, (4, 3)))
ax.set_ylim(-0.75, len(ks) - 0.5)
ax.text(1.01, -0.7, "global mean = 1", color=INK2, fontsize=9, va="bottom")
for i, k in enumerate(ks):
    ax.text(fac[k] + 0.02, i, f"{fac[k]:.2f}x  ({(fac[k]-1)*100:+.0f}%)", va="center", color=INK, fontsize=10.5)
ax.set_yticks(range(len(ks)), [NAMES[k] for k in ks]); ax.grid(axis="y", visible=False)
ax.set_xlim(0, 1.5)
ax.set(title="New York sea-level change per mm of global contribution", xlabel="Local / global ratio")
fig.tight_layout(); save(fig, "4_new_york_factor"); plt.show()

## 5. Total ice-sheet contribution - and a known problem with the generated total
`facts-total` adds every bamber19 output it is handed. The module also emits **AIS** (all of Antarctica), which already
*equals* East + West Antarctica, so the generated total counts Antarctica twice. The correct total is **AIS + GIS**.
Both are shown so the size of the error is visible.

In [ ]:
tot_y, tot_gen = slr("facts-total", "all-modules_global_total.nc")
correct = ais_g + gis_g
fig, ax = plt.subplots(figsize=(8, 4))
fan(ax, tot_y, correct, TOTAL, lw=2.4)
pg = np.percentile(tot_gen, 50, axis=0)
ax.plot(tot_y, pg, color=INK2, lw=2, ls=(0, (5, 3)))
pc = np.percentile(correct, 50, axis=0)
ax.annotate(f"Correct (AIS + GIS)\n{pc[-1]:.0f} mm", (tot_y[-1], pc[-1]), xytext=(-8, -34),
            textcoords="offset points", ha="right", fontsize=10, color=INK,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.9))
ax.annotate(f"facts-total as generated\n{pg[-1]:.0f} mm  (+{(pg[-1]/pc[-1]-1)*100:.0f}%)", (tot_y[-1], pg[-1]),
            xytext=(-8, 10), textcoords="offset points", ha="right", fontsize=10, color=INK2,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.9))
ax.set(title="Total ice-sheet contribution, global mean", ylabel="Sea-level contribution (mm)")
ax.set_xlim(tot_y[0], tot_y[-1])
ax.legend(handles=[Line2D([], [], color=TOTAL, lw=2.4, label="Correct total: median, 17-83%, 5-95%"),
                   Line2D([], [], color=INK2, lw=2, ls=(0, (5, 3)), label="Generated total: median")],
          loc="upper left")
fig.tight_layout(); save(fig, "5_total"); plt.show()

## 6. Run card
Timing and validation recorded by `run_bam.py` in `bam_results.json` (written next to the workspace).

In [ ]:
res = json.load(open(WS / "bam_results.json"))
run, outs = res.get("run", {}), res.get("outputs", {})
rows = [("exit code", run.get("exit")), ("wall time (s)", run.get("wall_s")), ("peak memory (MB)", run.get("max_rss_mb")),
        ("images pulled", run.get("images_pulled")),
        *[(f"service: {k}", f"{v}s") for k, v in run.get("service_s", {}).items()],
        ("output validation", "OK" if outs.get("ok") else "FAILED"),
        ("AIS - (EAIS+WAIS), max abs mm", outs.get("AIS_minus_EAIS_WAIS_maxabs_mm"))]
display(pd.DataFrame(rows, columns=["metric", "value"]).set_index("metric"))
for w in outs.get("warnings", []): print("WARNING:", w)